In [1]:
using LowLevelFEM

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (cache misses: include_dependency fsize change (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
n_load = 20
n_supp = 20

n_load -= 1

19

In [3]:
structured_rect_mesh()

mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=2, field=:u, rhs_field=:f)

Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f)

In [4]:
E = mat.E
ν = mat.ν
ρ = mat.ρ

7.85e-9

In [5]:
D = E / (1 + ν^2) * [1 ν 0; ν 1 0; 0 0 (1+ν)/2]

3×3 Matrix{Float64}:
     1.83486e5  55045.9        0.0
 55045.9            1.83486e5  0.0
     0.0            0.0        1.19266e5

In [6]:
K = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu))
M = ∫(Pu ⋅ Pu * ρ)

sparse([1, 9, 79, 81, 2, 10, 80, 82, 3, 25  …  241, 6, 42, 44, 46, 48, 222, 224, 240, 242], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3  …  241, 242, 242, 242, 242, 242, 242, 242, 242, 242], [8.722222222222218e-12, 4.36111111111111e-12, 4.36111111111111e-12, 2.180555555555555e-12, 8.722222222222218e-12, 4.36111111111111e-12, 4.36111111111111e-12, 2.180555555555555e-12, 8.72222222222222e-12, 4.361111111111112e-12  …  3.488888888888887e-11, 2.180555555555555e-12, 2.1805555555555562e-12, 8.72222222222222e-12, 8.72222222222222e-12, 2.180555555555555e-12, 2.180555555555554e-12, 8.72222222222222e-12, 8.722222222222218e-12, 3.488888888888887e-11], 242, 242)

In [7]:
using SparseArrays
n, n = size(M.A)
A = spzeros(n, n)
for i in 1:n
    A[i, i] = sum(M.A[i, :])
end
M = SystemMatrix(A, M.model, M.test_model, M.problems, M.offsets)

sparse([1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  233, 234, 235, 236, 237, 238, 239, 240, 241, 242], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  233, 234, 235, 236, 237, 238, 239, 240, 241, 242], [1.9624999999999997e-11, 1.9624999999999997e-11, 1.9625e-11, 1.9625e-11, 1.962499999999999e-11, 1.962499999999999e-11, 1.962499999999999e-11, 1.962499999999999e-11, 3.9249999999999994e-11, 3.9249999999999994e-11  …  7.85e-11, 7.85e-11, 7.849999999999997e-11, 7.849999999999997e-11, 7.849999999999999e-11, 7.849999999999999e-11, 7.85e-11, 7.85e-11, 7.849999999999996e-11, 7.849999999999996e-11], 242, 242)

In [8]:
f = VectorField[]
for i in 0:n_load
    gx = ScalarField(Pu, "right", (x, y, z) -> 100i)
    gy = ScalarField(Pu, "right", (x, y, z) -> 0)
    g = [gx, gy]
    f0 = ∫(Pu ⋅ g, Γ="right")
    push!(f, f0)
end
f = mergeFields(f)

nodal VectorField
[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0]

In [9]:
h = ScalarField(Pu, "left", (x, y, z, t) -> t / 10000, steps=n_supp)
bc1 = BoundaryCondition("left", problem=Pu, ux=h)
bc2 = BoundaryCondition("bottom", problem=Pu, uy=0)

BoundaryCondition("bottom", Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f), Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0))

In [10]:
λmax = smallestPeriodTime(K, M, support=[bc1, bc2])
dt = λmax / π

2.0513041788025482e-8

In [11]:
u0 = VectorField(Pu, "body", [0, 0, 0])
u0 = elementsToNodes(u0)
u0 = projectTo2D(u0)

v0 = VectorField(Pu, "body", [0, 0, 0])
v0 = elementsToNodes(v0)
v0 = projectTo2D(v0)

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [12]:
u, v = HHT(K, M, f, u0, v0, 20, dt, support=[bc1, bc2])

(VectorField(Matrix{Float64}[], [0.0 0.0001 … 0.0018 0.0019; 0.0 0.0 … 0.0 0.0; … ; 0.0 3.1821155175385616e-6 … 0.009974775318601558 0.01116918325403425; 0.0 -1.5156025735254636e-7 … -0.0016969377933342379 -0.0019302065878905358], [0.0, 2.0513041788025482e-8, 4.1026083576050964e-8, 6.153912536407645e-8, 8.205216715210193e-8, 1.025652089401274e-7, 1.230782507281529e-7, 1.435912925161784e-7, 1.6410433430420388e-7, 1.8461737609222937e-7, 2.0513041788025486e-7, 2.2564345966828036e-7, 2.4615650145630585e-7, 2.6666954324433134e-7, 2.8718258503235683e-7, 3.076956268203823e-7, 3.282086686084078e-7, 3.487217103964333e-7, 3.692347521844588e-7, 3.897477939724843e-7], Int64[], 20, :v2D, Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f)), VectorField(Matrix{

In [13]:
showDoFResults(u, name="u", visible=true, factor=10)
showDoFResults(v, name="v", visible=false)
showDoFResults(f, name="f")

2

In [14]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
